# 🎥 03 — Live Camera Detection

Real-time detection using your webcam. This notebook:
1. Tests the camera feed
2. Runs a live detection loop with FPS overlay
3. Optionally records a short annotated clip

> ⚠️ **Requires a webcam.** If running on a remote server / Colab, use `02_image_detection` instead.

In [ ]:
# ── Imports & Setup ───────────────────────────────────────────────────
import sys, os, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

import cv2
import numpy as np
from IPython.display import display, Image as IPImage, clear_output

from detector import ObjectDetector
from utils import compute_heuristic_measurements, classify_severity

MODEL_PATH = PROJECT_ROOT / "models" / "livedet_best.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "live_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

detector = ObjectDetector(model_path=str(MODEL_PATH), device="cpu", conf=0.25)
print(f"✅ Detector ready — classes: {detector.class_names}")

In [ ]:
# ── Camera Quick Test ─────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("❌ Cannot open camera 0. Check your webcam connection.")

ret, test_frame = cap.read()
cap.release()

if ret:
    # Show the test frame inside the notebook
    _, buf = cv2.imencode(".jpg", test_frame)
    display(IPImage(data=buf.tobytes(), width=480))
    print(f"✅ Camera OK — frame shape: {test_frame.shape}")
else:
    print("⚠️ Camera opened but failed to grab a frame.")

In [ ]:
# ── Live Detection Loop (OpenCV Window) ──────────────────────────────
# Press 'q' to quit, 's' to save a snapshot.
# This cell opens an OpenCV window — works on local machines only.

RESIZE_W = 640

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open camera")

frame_count = 0
fps = 0.0
prev_time = time.time()

print("🎥 Live detection started — press 'q' to stop, 's' to snapshot")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Resize for speed
        h, w = frame.shape[:2]
        scale = RESIZE_W / w
        frame = cv2.resize(frame, (RESIZE_W, int(h * scale)))

        # Detect
        result = detector.detect(frame)
        annotated = detector.annotate_image(frame.copy(), result["detections"])

        # FPS counter
        frame_count += 1
        now = time.time()
        if now - prev_time >= 1.0:
            fps = frame_count / (now - prev_time)
            frame_count = 0
            prev_time = now

        # HUD overlay
        cv2.putText(annotated, f"FPS: {fps:.1f}", (10, 28),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(annotated, f"Detections: {result['total_detections']}", (10, 56),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        cv2.imshow("LIVEDET — Live Detection", annotated)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("s"):
            snap_path = OUTPUT_DIR / f"snapshot_{int(time.time())}.jpg"
            cv2.imwrite(str(snap_path), annotated)
            print(f"  📸 Snapshot saved: {snap_path}")

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Camera released.")

In [ ]:
# ── Record a Short Annotated Clip (10 seconds) ──────────────────────
RECORD_SECONDS = 10
CLIP_PATH = str(OUTPUT_DIR / "live_clip.mp4")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open camera")

ret, sample = cap.read()
h, w = sample.shape[:2]
scale = RESIZE_W / w
out_w, out_h = RESIZE_W, int(h * scale)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(CLIP_PATH, fourcc, 20.0, (out_w, out_h))

start = time.time()
frames_written = 0
print(f"🔴 Recording {RECORD_SECONDS}s clip to {CLIP_PATH} ...")

while time.time() - start < RECORD_SECONDS:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (out_w, out_h))
    result = detector.detect(frame)
    annotated = detector.annotate_image(frame.copy(), result["detections"])
    writer.write(annotated)
    frames_written += 1

cap.release()
writer.release()
print(f"✅ Saved {frames_written} frames ({frames_written/RECORD_SECONDS:.1f} FPS) → {CLIP_PATH}")

## 🛠 Troubleshooting

| Problem | Fix |
|---------|-----|
| Camera index wrong | Change `cv2.VideoCapture(0)` to `1`, `2`, etc. |
| Low FPS | Reduce `RESIZE_W` to `480` or `320` |
| Black frames | Check lighting or try a different camera backend: `cv2.VideoCapture(0, cv2.CAP_DSHOW)` |
| "Cannot open camera" on server | Use `02_image_detection` notebook for headless environments |

### Alternative: Browser-based live detection
Run the WebSocket server instead:
```bash
cd backend
python live_ws.py
```
Then open the **Live Camera** tab in the frontend UI.